# Evaluation Dataset Check

Quick notebook to verify JSON and CSV datasets in `backend/evaluation/dataset`.
It checks file presence, row/column counts, basic quality indicators, and previews sample rows.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'playground' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATASET_DIR = ROOT / 'backend' / 'evaluation' / 'dataset'
json_files = sorted(DATASET_DIR.glob('*.json'))
csv_files = sorted(DATASET_DIR.glob('*.csv'))

print('DATASET_DIR:', DATASET_DIR)
print('JSON files:', [p.name for p in json_files])
print('CSV files :', [p.name for p in csv_files])

if not json_files:
    raise FileNotFoundError('No JSON files found in backend/evaluation/dataset')

DATASET_DIR: /home/aman/storage/Python/Projects/Research Paper Assistant/backend/evaluation/dataset
JSON files: ['export_for_annotation.json', 'qa_pairs.json']
CSV files : ['export_for_annotation.csv', 'qa_pairs.csv']


In [2]:
def to_df_from_json(path: Path) -> pd.DataFrame:
    data = json.loads(path.read_text(encoding='utf-8'))
    if isinstance(data, list):
        return pd.json_normalize(data)
    if isinstance(data, dict):
        return pd.json_normalize(data)
    return pd.DataFrame({'value': [data]})

summary_rows = []
for jf in json_files:
    cf = jf.with_suffix('.csv')
    jdf = to_df_from_json(jf)
    cdf = pd.read_csv(cf) if cf.exists() else pd.DataFrame()

    summary_rows.append({
        'dataset': jf.stem,
        'json_rows': len(jdf),
        'json_cols': len(jdf.columns),
        'csv_exists': cf.exists(),
        'csv_rows': len(cdf),
        'csv_cols': len(cdf.columns),
        'row_count_match': len(jdf) == len(cdf) if cf.exists() else False,
        'columns_match': set(jdf.columns) == set(cdf.columns) if cf.exists() else False,
        'null_cells_csv': int(cdf.isna().sum().sum()) if cf.exists() else None,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,dataset,json_rows,json_cols,csv_exists,csv_rows,csv_cols,row_count_match,columns_match,null_cells_csv
0,export_for_annotation,21,17,True,21,17,True,True,42
1,qa_pairs,32,9,True,32,9,True,True,0


In [4]:
for jf in json_files:
    cf = jf.with_suffix('.csv')
    print()
    print('=' * 80)
    print(f'Dataset: {jf.name}')
    print('-' * 80)

    jdf = to_df_from_json(jf)
    print('JSON shape:', jdf.shape)
    display(jdf.head(3))

    if cf.exists():
        cdf = pd.read_csv(cf)
        print('CSV shape :', cdf.shape)
        display(cdf.head(3))

        if 'question' in cdf.columns:
            q_dup = int(cdf['question'].astype(str).duplicated().sum())
            print('Duplicate questions in CSV:', q_dup)
    else:
        print('CSV missing for this dataset.')


Dataset: export_for_annotation.json
--------------------------------------------------------------------------------
JSON shape: (21, 17)


,paper_id,paper_type,document_id,section_id,section_title,resolved_section_title,guide_step_number,guide_step_title,guide_step_description,guide_questions,chunks,figure_context,figure_chunk_ids,table_context,table_chunk_ids,needs_figures,needs_tables
0,paper_theory,Theory,30c88170-fd15-5486-bf70-bbab16747183,30c88170-fd15-5486-bf70-bbab16747183_section_20,1 Introduction,"Irreducible representations πλ ( g ) i,j",1,Step 1: Understand the introduction and contex...,Understand the introduction and context of the...,[What is the primary task in quantum informati...,[{'chunk_id': '0b5bdfe8-58af-5258-acfd-1485126...,,[],,[],False,False
1,paper_theory,Theory,30c88170-fd15-5486-bf70-bbab16747183,30c88170-fd15-5486-bf70-bbab16747183_section_19,1.1 Our contributions,The trace function,2,Step 2: Identify the main contributions of the...,Identify the main contributions of the paper,[What is the significance of the unified frame...,[{'chunk_id': 'fc5d219e-8a69-54cd-b8fa-fe10c1d...,,[],,[],False,False
2,paper_theory,Theory,30c88170-fd15-5486-bf70-bbab16747183,30c88170-fd15-5486-bf70-bbab16747183_section_5,2 Preliminaries,Organization of the paper,1,Step 1: Understand the preliminary concepts an...,Understand the preliminary concepts and notations,[What is the significance of the unitary group...,[{'chunk_id': 'b60e7098-8b01-573e-902c-d0f1b4d...,,[],,[],False,False


CSV shape : (21, 17)


,paper_id,paper_type,document_id,section_id,section_title,resolved_section_title,guide_step_number,guide_step_title,guide_step_description,guide_questions,chunks,figure_context,figure_chunk_ids,table_context,table_chunk_ids,needs_figures,needs_tables
0,paper_theory,Theory,30c88170-fd15-5486-bf70-bbab16747183,30c88170-fd15-5486-bf70-bbab16747183_section_20,1 Introduction,"Irreducible representations πλ ( g ) i,j",1,Step 1: Understand the introduction and contex...,Understand the introduction and context of the...,['What is the primary task in quantum informat...,[{'chunk_id': '0b5bdfe8-58af-5258-acfd-1485126...,NaN,[],NaN,[],False,False
1,paper_theory,Theory,30c88170-fd15-5486-bf70-bbab16747183,30c88170-fd15-5486-bf70-bbab16747183_section_19,1.1 Our contributions,The trace function,2,Step 2: Identify the main contributions of the...,Identify the main contributions of the paper,['What is the significance of the unified fram...,[{'chunk_id': 'fc5d219e-8a69-54cd-b8fa-fe10c1d...,NaN,[],NaN,[],False,False
2,paper_theory,Theory,30c88170-fd15-5486-bf70-bbab16747183,30c88170-fd15-5486-bf70-bbab16747183_section_5,2 Preliminaries,Organization of the paper,1,Step 1: Understand the preliminary concepts an...,Understand the preliminary concepts and notations,['What is the significance of the unitary grou...,[{'chunk_id': 'b60e7098-8b01-573e-902c-d0f1b4d...,NaN,[],NaN,[],False,False



Dataset: qa_pairs.json
--------------------------------------------------------------------------------
JSON shape: (32, 9)


,paper_id,paper_type,document_id,section_id,section_title,question,reference_answer,relevant_chunk_ids,question_type
0,paper_theory,Theory,30c88170-fd15-5486-bf70-bbab16747183,30c88170-fd15-5486-bf70-bbab16747183_section_19,1.1 Our contributions,What is the upper bound on the query complexit...,Proposition 8 states that for any unitary irre...,"[f87fdaf6-6569-5ec1-8c3c-baaa5b533e01, e7a075b...",factual
1,paper_theory,Theory,30c88170-fd15-5486-bf70-bbab16747183,30c88170-fd15-5486-bf70-bbab16747183_section_19,1.1 Our contributions,Which mathematical duality is used in the proo...,The proof applies Schur-Weyl duality to decomp...,"[6a3bd22d-f0c4-520d-bff6-85f9ddc0178e, e7a075b...",conceptual
2,paper_theory,Theory,30c88170-fd15-5486-bf70-bbab16747183,30c88170-fd15-5486-bf70-bbab16747183_section_19,1.1 Our contributions,What is the role of the partial transpose oper...,The partial transpose is taken over systems BE...,"[5e934d2e-aafa-5ed8-8301-47b3eed26318, 16a5c6f...",conceptual


CSV shape : (32, 9)


,paper_id,paper_type,document_id,section_id,section_title,question,reference_answer,relevant_chunk_ids,question_type
0,paper_theory,Theory,30c88170-fd15-5486-bf70-bbab16747183,30c88170-fd15-5486-bf70-bbab16747183_section_19,1.1 Our contributions,What is the upper bound on the query complexit...,Proposition 8 states that for any unitary irre...,"['f87fdaf6-6569-5ec1-8c3c-baaa5b533e01', 'e7a0...",factual
1,paper_theory,Theory,30c88170-fd15-5486-bf70-bbab16747183,30c88170-fd15-5486-bf70-bbab16747183_section_19,1.1 Our contributions,Which mathematical duality is used in the proo...,The proof applies Schur-Weyl duality to decomp...,"['6a3bd22d-f0c4-520d-bff6-85f9ddc0178e', 'e7a0...",conceptual
2,paper_theory,Theory,30c88170-fd15-5486-bf70-bbab16747183,30c88170-fd15-5486-bf70-bbab16747183_section_19,1.1 Our contributions,What is the role of the partial transpose oper...,The partial transpose is taken over systems BE...,"['5e934d2e-aafa-5ed8-8301-47b3eed26318', '16a5...",conceptual


Duplicate questions in CSV: 0


## Notes

- If `row_count_match` is false, regenerate the corresponding CSV.
- If `columns_match` is false, check whether JSON has nested structures that need special flattening.